# End-to-end walkthrough — speech emotion recognition

**What this notebook is.** The complete record of this project, in the order it
happened, with the code that produced every number. It covers which data was
used, how it was split into training and test sets, every method that was tried,
what worked, what did not, and why.

Nothing here is quoted from memory. Each result is either recomputed when the
notebook runs, or read back from the logged run that produced it, so every
figure can be traced to the code that generated it.

**Two things to take from here if you take nothing else:**

1. **`splits/folds.csv` is frozen.** It records exactly which actors belong to
   the training side and which to the test side of every fold. Any new experiment
   must read it rather than make its own split, because two results computed on
   different splits are not comparable — a model that looks two points better may
   simply have been handed easier test speakers. The file has not changed since
   22 August 2026, and an automated test fails if it is ever edited.

2. **Score through `ser.evaluate`.** It computes accuracy, macro-F1, the
   confusion matrix and the per-gender, per-intensity and per-emotion breakdowns
   from one place, and appends a row to `results/results.csv`. Using it means
   every result in the project was measured the same way. A run that is not
   logged there cannot be compared against anything, so in practice it did not
   happen.

**Headline results**

| Test condition | Accuracy | Macro-F1 |
|---|---|---|
| Unseen speakers, same corpus (RAVDESS) | 0.824 | **0.822** |
| All four corpora used, frozen-backbone transfer | 0.785 | 0.782 |
| Unseen speakers, normal (non-acted) delivery | 0.768 | 0.681 |
| **Unseen corpus entirely** | — | **0.380** |

The headline configuration is chosen by **nested cross-validation** (§10), so it
carries no selection bias from having been picked on the test folds.

The gap between the first and last row is the most important finding in the
project. More on that in §8.

## Setup

In [1]:
# Run from anywhere: this notebook lives in notebooks/, the package in ser/.
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "ser").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)          # so relative paths like features/ resolve
print("project root:", ROOT)

import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from ser import SEED, set_seeds
set_seeds()                     # numpy, python random, torch if available
print("seed fixed at", SEED)

project root: C:\dev\Unheard


seed fixed at 42


---
## 1. The data

**RAVDESS** — 1,440 clips, 24 actors, 8 emotions. Actors read two fixed
sentences ("Kids are talking by the door", "Dogs are sitting by the door") in
each emotion, at two intensity levels, twice.

`ser.metadata` is the single source of truth for *which clips exist*. It parses
the filename and never opens the audio.

In [2]:
from ser.metadata import load as load_metadata

meta = load_metadata()
print("clips:", len(meta), "| actors:", meta.actor.nunique())
meta[["filename","emotion_label","intensity_label","statement_label","actor","gender"]].head()

clips: 1440 | actors: 24


,filename,emotion_label,intensity_label,statement_label,actor,gender
0,01-01-01-01-01.wav,neutral,normal,kids,1,male
1,01-01-01-02-01.wav,neutral,normal,kids,1,male
2,01-01-02-01-01.wav,neutral,normal,dogs,1,male
3,01-01-02-02-01.wav,neutral,normal,dogs,1,male
4,02-01-01-01-01.wav,calm,normal,kids,1,male


The filename encodes everything. `03-01-05-02-01-01-12.wav` in the canonical
release; **this copy has 5 fields, not 7** — the constant modality and
vocal-channel fields were stripped when it was repackaged. The parser *rejects*
7-field names rather than silently mis-parsing them by two positions.

In [3]:
print(meta.emotion_label.value_counts().reindex(
    ["neutral","calm","happy","sad","angry","fearful","disgust","surprised"]).to_string())
print("\nneutral has half the clips of every other emotion:")
print("  there is no strong-intensity neutral recording — it isn't a thing.")
print("  strong-intensity neutral clips found:",
      int(((meta.emotion=='01') & (meta.intensity=='02')).sum()))

emotion_label
neutral       96
calm         192
happy        192
sad          192
angry        192
fearful      192
disgust      192
surprised    192

neutral has half the clips of every other emotion:
  there is no strong-intensity neutral recording — it isn't a thing.
  strong-intensity neutral clips found: 0


### What we verified about the audio (Phase 1 audit)

Every fact below was measured, not taken from documentation — and two of them
contradicted it.

In [4]:
audit = pd.DataFrame([
    ["files",                 "1,440",        "as documented"],
    ["sample rate",           "48,000 Hz",    "uniform — but librosa defaults to 22,050, so we always pass sr="],
    ["bit depth",             "PCM_16",       "uniform"],
    ["channels",              "1,435 mono + 5 STEREO", "⚠ docs said all mono; both channels are bit-identical so averaging is lossless"],
    ["filename fields",       "5, not 7",     "⚠ repackaged copy; parser rejects 7-field names"],
    ["duration",              "2.94–5.27 s",  "median 3.67"],
    ["corrupt / zero-length", "0",            "every file opened"],
], columns=["property","value","note"])
audit

,property,value,note
0,files,"1,440",as documented
1,sample rate,"48,000 Hz","uniform — but librosa defaults to 22,050, so w..."
2,bit depth,PCM_16,uniform
3,channels,"1,435 mono + 5 STEREO",⚠ docs said all mono; both channels are bit-id...
4,filename fields,"5, not 7",⚠ repackaged copy; parser rejects 7-field names
5,duration,2.94–5.27 s,median 3.67
6,corrupt / zero-length,0,every file opened


**Project rule that came out of this:** never call `librosa.load` directly.
`ser.preprocess.load_audio` is the one entry point — it applies the mono
downmix and the 16 kHz target rate, so those decisions hold everywhere instead
of being re-decided at each call site.

---
## 2. The external corpora

Three further public emotion-speech datasets were combined with RAVDESS, giving
12,162 clips instead of 1,440.

"Combining" them is not simply concatenation. Each dataset labels emotions with
its own vocabulary and its own codes, so every one had to be mapped onto the
eight labels RAVDESS uses, and any clip whose label had no equivalent had to be
excluded rather than guessed at. `ser.corpora` performs that mapping and refuses
to run if it ever meets a code it does not recognise — failing loudly is safer
than silently mislabelling a clip.

In [5]:
from ser import corpora

index = corpora.load_index()
print(corpora.summarise(index).to_string(index=False))

 source  clips  speakers  emotions  male  female  unknown_sex
RAVDESS   1440        24         8   720     720            0
CREMA-D   7442        91         6  3930    3512            0
   TESS   2800         2         7     0    2800            0
  SAVEE    480         4         7   480       0            0
  TOTAL  12162       121         8  5130    7032            0


The shape of the problem is already visible in that table:

- **CREMA-D** has only 6 emotions — no `calm`, no `surprised`.
- **TESS** is 2,800 clips from just **two** speakers.
- **SAVEE** is four men, and its audio is band-limited around 3 kHz where the
  others reach 5.7 kHz — a model can identify it trivially.
- **`calm` exists only in RAVDESS**, so on a merged test set a model could score
  on `calm` just by recognising the corpus.

In [6]:
print(pd.crosstab(index.emotion_label, index.source).reindex(
    ["neutral","calm","happy","sad","angry","fearful","disgust","surprised"]).to_string())

source         RAVDESS  CREMA-D  TESS  SAVEE
emotion_label                               
neutral             96     1087   400    120
calm               192        0     0      0
happy              192     1271   400     60
sad                192     1271   400     60
angry              192     1271   400     60
fearful            192     1271   400     60
disgust            192     1271   400     60
surprised          192        0   400     60


---
## 3. Train and test — the part that matters most

**The tempting split:** shuffle all 1,440 clips, take 80% to train. Actor 07
lands in both halves. The model learns *"actor 07's angry voice sounds like
this"* and scores brilliantly on actor 07's other clips. It has learned the
**person**, not the emotion.

**What we do:** split by **actor**. The 24 actors go into 5 groups; train on 4
groups, test on the 5th — voices the model has never heard. Repeat 5 times so
every actor is tested exactly once.

This is `speaker_independent`, the primary protocol. `splits/folds.csv` records
the assignment and is **frozen**.

In [7]:
from ser.splits import load_folds, get_split

folds = load_folds()
lookup = meta.set_index("filename")

rows = []
for f in range(5):
    tr, te = get_split("speaker_independent", f, folds=folds)
    rows.append({
        "fold": f,
        "train_clips": len(tr), "test_clips": len(te),
        "train_actors": lookup.loc[tr,"actor"].nunique(),
        "test_actors":  lookup.loc[te,"actor"].nunique(),
        "ACTOR OVERLAP": len(set(lookup.loc[tr,"actor"]) & set(lookup.loc[te,"actor"])),
    })
fold_table = pd.DataFrame(rows)
print(fold_table.to_string(index=False))
print("\ntest clips summed over folds:", fold_table.test_clips.sum(),
      "— every clip tested exactly once")
print("fold 4 is smaller because 24 actors do not divide by 5")

 fold  train_clips  test_clips  train_actors  test_actors  ACTOR OVERLAP
    0         1140         300            19            5              0
    1         1140         300            19            5              0
    2         1140         300            19            5              0
    3         1140         300            19            5              0
    4         1200         240            20            4              0

test clips summed over folds: 1440 — every clip tested exactly once
fold 4 is smaller because 24 actors do not divide by 5


**Pooled scoring, not averaged.** There are two ways to turn five fold results
into one number, and they do not agree.

The obvious way is to score each fold separately and average the five scores.
The problem is that the folds are unequal — four hold 300 test clips and the
fifth holds 240, because 24 actors do not divide evenly by 5. Averaging the five
scores treats the 240-clip fold as equally important as each 300-clip fold, so
every clip in the small fold silently counts for about 25% more than every clip
in the others.

The alternative, used here, is to collect the held-out prediction for all 1,440
clips and score them together, once. Every clip counts exactly the same. This
matters most for macro-F1, which depends on per-class scores, and the rarest
class — neutral — has as few as 16 clips in a single fold.

### Four protocols exist; we report the first and use the second only as a check

In [8]:
from ser.splits import PROTOCOLS

blurb = {
    "speaker_independent":  "PRIMARY. No actor appears on both sides.",
    "random_stratified":    "Leakage check ONLY. Same actors both sides.",
    "statement_holdout":    "Train on sentence 1, test on sentence 2.",
    "statement_holdout_si": "Same, but speakers held out too.",
}
for p in PROTOCOLS:
    n = folds[folds.protocol == p].fold.nunique()
    print(f"  {p:<22s} {n} fold(s)   {blurb[p]}")

  speaker_independent    5 fold(s)   PRIMARY. No actor appears on both sides.
  random_stratified      5 fold(s)   Leakage check ONLY. Same actors both sides.
  statement_holdout      1 fold(s)   Train on sentence 1, test on sentence 2.
  statement_holdout_si   5 fold(s)   Same, but speakers held out too.


### How much does the easy split inflate results? We measured it.

In [9]:
res = pd.read_csv("results/results.csv")
pooled = res[res.fold.astype(str) == "pooled"]
cfg = "mfcc20-d-dd-mean_std-trim-chroma+spectral_centroid+spectral_contrast+zcr-per_speaker"

gap = (pooled[(pooled.feature_config == cfg) & (pooled.model == "svm-rbf")]
       .groupby("protocol")[["accuracy","macro_f1"]].max()
       .loc[["speaker_independent","random_stratified"]])
gap.loc["INFLATION"] = gap.loc["random_stratified"] - gap.loc["speaker_independent"]
print(gap.round(4).to_string())
print("\nShuffling instead of splitting by actor hands you ~8.8 free points")
print("that would vanish on a real unseen speaker.")

                     accuracy  macro_f1
protocol                               
speaker_independent    0.6389    0.6311
random_stratified      0.7271    0.7200
INFLATION              0.0882    0.0889

Shuffling instead of splitting by actor hands you ~8.8 free points
that would vanish on a real unseen speaker.


---
## 4. Features — what we extract from the audio

Raw audio is millions of numbers. Features reduce each clip to a few hundred
meaningful ones. **Every parameter is explicit in `FeatureConfig`** — none is
left to a library default, because an unstated default is one nobody can see,
question, or sweep.

In [10]:
from ser.features import FeatureConfig
from ser.run_phase5 import WINNER as HANDCRAFTED

print(HANDCRAFTED.describe())

name            mfcc20-d-dd-mean_std-trim-chroma+spectral_centroid+spectral_contrast+zcr-per_speaker
hash            4a95787c2818
sample rate     16000 Hz
window          2048 samples = 128.0 ms
hop             512 samples = 32.0 ms
overlap         75%
frame rate      31.2 frames/s
mel bands       128  (0 Hz to 8000 Hz)
MFCCs           20 +delta +delta2
delta width     9 frames = 288 ms of context
aggregation     mean_std
trim silence    True (top_db=30)
normalisation   per_speaker


**What those mean, briefly.** MFCCs describe the shape of the sound spectrum on
a scale matched to human hearing. Deltas are how fast those change. `mean_std`
collapses the time axis into an average and a spread. `per_speaker`
normalisation is the important one — see below.

**Phase 4 swept 19 configurations one axis at a time.** Findings:
- per-speaker normalisation: **+6.53 pp** — the largest single gain
- a "corrected" 25 ms framing window that theory says should help: **−1.25 pp**,
  a negative result we kept rather than hid

### Per-speaker normalisation — and the one leakage exception we allow

Everyone's voice has a baseline. One person speaks at a naturally higher pitch
than another; one recording is louder than another because of where the
microphone sat. None of that is emotion, but all of it ends up in the numbers.

Per-speaker normalisation removes it. For each speaker, take all of their clips,
compute the average and spread of every feature across just those clips, and
re-express each of their clips relative to their own averages. A clip is then
described as *"higher pitched than this person usually is"* rather than
*"440 Hz"*. What is left is what varies within a voice — which is where emotion
lives.

**This is transductive and we say so.** Under `speaker_independent` a held-out
speaker has no training clips by construction, so their statistics come from
their own held-out clips. It consumes **no labels** — only audio, which you
would also have at deployment time — and it is standard practice (CMVN). It is
the *only* stated exception in the project, and it is documented in CLAUDE.md
rule 2 rather than buried.

---
## 5. Classical baselines — seven models, identical data

All seven on the same features and the same folds, so the comparison isolates
the estimator.

In [11]:
best = (pooled[(pooled.protocol == "speaker_independent") &
               (pooled.feature_config == cfg)]
        .groupby("model")[["accuracy","macro_f1"]].max()
        .sort_values("macro_f1", ascending=False))
print(best.round(4).to_string())

                           accuracy  macro_f1
model                                        
svm-rbf-tuned                0.6389    0.6333
svm-rbf                      0.6389    0.6311
svm-rbf-two-stage-arousal    0.6194    0.6149
xgboost                      0.6042    0.5956
logistic-regression          0.5931    0.5925
linear-svc                   0.5722    0.5641
random-forest                0.5688    0.5488
gaussian-nb                  0.5160    0.5071
knn                          0.4861    0.4602
adaboost                     0.4507    0.4504


SVM-RBF won at **0.6389 / 0.6311**. Tuning it via grid search *inside* the
training folds then bought **+0.22 pp** — which later turned out to be inside
the noise (§11). Feature design moved the number 13 points; estimator tuning
moved it nothing.

---
## 6. wav2vec2 — the jump

**What it is.** A model Meta trained on 960 hours of speech with *no labels* —
it learned by hiding slices of audio and predicting them. It has never seen an
emotion label. We **freeze** it (no retraining), push each clip through, and
read the numbers out.

**It has 12 internal layers**, each producing 768 numbers per 20 ms of audio.
You must choose which to read from — and this is where the result is.

In [12]:
from ser.embeddings import EmbeddingConfig
print(EmbeddingConfig(layer=6, pooling="mean", normalisation="per_speaker").describe())

name            wav2vec2-base-L6-mean-per_speaker
hash            642607c7a846
model           facebook/wav2vec2-base
                wav2vec2-base, 12 layers, 768-d, 95M params
sample rate     16000 Hz
layer           6
pooling         mean
max duration    whole clip
peak norm       False
trim silence    False (top_db=30)
normalisation   per_speaker (applied per fold by the caller)
cache           w2v_642607c7a846.npy


In [13]:
curve = pd.read_parquet("features/phase13c_layer_curve.parquet")
pivot = curve.pivot(index="layer", columns="normalisation", values="macro_f1").round(4)
print(pivot.to_string())
print("\nbest layer:", int(pivot.per_speaker.idxmax()),
      "at", round(pivot.per_speaker.max(), 4))
print("last layer (what every tutorial uses):", round(pivot.per_speaker.loc[12], 4))
print("cost of following the default: %.1f points" %
      (100*(pivot.per_speaker.max() - pivot.per_speaker.loc[12])))

normalisation  global  per_speaker
layer                             
0              0.4994       0.6173
1              0.5645       0.6731
2              0.6096       0.7017
3              0.6417       0.7503
4              0.6569       0.7466
5              0.6628       0.7703
6              0.6758       0.7856
7              0.6751       0.7740
8              0.6559       0.7603
9              0.6331       0.7453
10             0.6003       0.7157
11             0.5059       0.6212
12             0.4999       0.5942

best layer: 6 at 0.7856
last layer (what every tutorial uses): 0.5942
cost of following the default: 19.1 points


**Emotion lives in the middle of the network.** wav2vec2's training rewards
predicting *what word* comes next, so its upper layers specialise in words and
discard everything else. Emotion is in the delivery, and that survives mid-stack.
Layers 11 and 12 are *worse than our hand-built MFCC features*.

### The full sweep, one axis at a time

In [14]:
sweep = pd.read_parquet("features/phase13_sweep.parquet")
print(sweep[["axis","config","model","accuracy","macro_f1","vs_floor_pp"]]
      .head(10).to_string(index=False))

         axis                            config               model  accuracy  macro_f1  vs_floor_pp
         head wav2vec2-base-L6-mean-per_speaker logistic-regression  0.799306  0.794382        16.33
normalisation wav2vec2-base-L6-mean-per_speaker             svm-rbf  0.789583  0.785612        15.45
         head wav2vec2-base-L6-mean-per_speaker             svm-rbf  0.789583  0.785612        15.45
      pooling      wav2vec2-base-L6-mean-global             svm-rbf  0.688194  0.675817         4.47
normalisation      wav2vec2-base-L6-mean-global             svm-rbf  0.688194  0.675817         4.47
        layer      wav2vec2-base-L6-mean-global             svm-rbf  0.688194  0.675817         4.47
      pooling  wav2vec2-base-L6-mean_std-global             svm-rbf  0.673611  0.665530         3.44
         head wav2vec2-base-L6-mean-per_speaker             xgboost  0.668750  0.660621         2.95
         head wav2vec2-base-L6-mean-per_speaker       random-forest  0.668056  0.655978    

**Winner: layer 6 · mean pooling · per-speaker normalisation · logistic
regression → 0.7993 / 0.7944.**

Note which head won. Logistic regression — a *linear* model — beat XGBoost and
random forest by 13 points. When the simplest classifier wins by that margin,
the features are doing the work.

*(Logistic regression predicts a category, not a number. It is not linear
regression.)*

### Reproducing the winning number from the cached embeddings

In [15]:
from ser.run_wav2vec2 import build_matrix, load_cached, ravdess_view
from ser.evaluate import evaluate_pooled
from sklearn.linear_model import LogisticRegression

tensor, idx = load_cached(EmbeddingConfig(batch_size=8))
tensor, idx = ravdess_view(tensor, idx)          # 12,162 clips -> RAVDESS 1,440
WIN = EmbeddingConfig(layer=6, pooling="mean", normalisation="per_speaker")
X, y = build_matrix(tensor, idx, WIN)
print("feature matrix:", X.shape)

per_fold = []
for f in range(5):
    tr, te = get_split("speaker_independent", f, folds=folds)
    clf = LogisticRegression(max_iter=2000, random_state=SEED)
    clf.fit(X.loc[tr].to_numpy(), y.loc[tr].to_numpy())
    pred = clf.predict(X.loc[te].to_numpy())
    per_fold.append((y.loc[te].to_numpy(), pred,
                     idx.set_index("filename").loc[te].reset_index()))

out = evaluate_pooled(per_fold, "speaker_independent", model="logistic-regression",
                      feature_config=WIN.name, notes="notebook-03 reproduction",
                      append=False)
print(f"\nreproduced: acc={out['accuracy']:.4f}  macro-F1={out['macro_f1']:.4f}")

feature matrix: (1440, 768)



reproduced: acc=0.7993  macro-F1=0.7944


---
## 7. Does adding the other three datasets help? No.

**How this was tested.** The test set stays *exactly* the frozen RAVDESS folds.
Only the training side grows. So the comparison is single-variable — same clips
scored, same scorer, only the training data differs. Zero external clips are
ever tested.

In [16]:
from ser.splits_merged import augmented_train
src = index.set_index("filename")["source"].astype(str)

rows = []
for f in range(5):
    tr, te = augmented_train(f, sources=("CREMA-D","TESS","SAVEE"), index=index)
    c = src.loc[tr].value_counts()
    rows.append({"fold": f, "TRAIN": len(tr),
                 "RAVDESS": c.get("RAVDESS",0), "CREMA-D": c.get("CREMA-D",0),
                 "TESS": c.get("TESS",0), "SAVEE": c.get("SAVEE",0),
                 "TEST": len(te),
                 "external in test": int((src.loc[te] != "RAVDESS").sum())})
print(pd.DataFrame(rows).to_string(index=False))

 fold  TRAIN  RAVDESS  CREMA-D  TESS  SAVEE  TEST  external in test
    0  11862     1140     7442  2800    480   300                 0
    1  11862     1140     7442  2800    480   300                 0
    2  11862     1140     7442  2800    480   300                 0
    3  11862     1140     7442  2800    480   300                 0
    4  11922     1200     7442  2800    480   240                 0


In [17]:
print(pd.read_parquet("features/phase12_corpus_sweep.parquet")
      .to_string(index=False))

             added  n_train_fold0  accuracy  macro_f1  seconds  vs_control_pp
    (ravdess only)           1140  0.638889  0.631106      0.7           0.00
             SAVEE           1620  0.621528  0.613138      1.0          -1.80
        TESS+SAVEE           4420  0.585417  0.580376      2.7          -5.07
              TESS           3940  0.582639  0.579442      2.1          -5.17
     CREMA-D+SAVEE           9062  0.540972  0.536053     19.6          -9.51
           CREMA-D           8582  0.534722  0.527517     17.7         -10.36
CREMA-D+TESS+SAVEE          11862  0.523611  0.520038     29.6         -11.11
      CREMA-D+TESS          11382  0.517361  0.512209     27.1         -11.89


**Every combination loses**, and the damage tracks the *volume* of outside data
(correlation −0.98). Four standard covariate-shift remedies were then tried —
down-weighting, density-ratio importance weighting, class-selective inclusion,
scaler-only. **None beat using no external data.** Inner cross-validation, given
the choice and no sight of the test fold, chose **zero external clips in all
five folds**.

---
## 8. Leave-one-corpus-out — the honest deployment number

Every protocol so far holds out *speakers* recorded in the same studio, on the
same equipment, reading the same prompts. This one holds out a **whole corpus**:
train on three, test on the fourth — different people, microphones, room and
sentences.

In [18]:
from ser.splits_merged import summarise_merged_folds
s = summarise_merged_folds()
print(s[s.protocol=="corpus_holdout"][
    ["fold","n_train","n_test","train_speakers","test_speakers",
     "speaker_overlap","test_corpora","test_calm"]].to_string(index=False))

 fold  n_train  n_test  train_speakers  test_speakers  speaker_overlap test_corpora  test_calm
    0    10722    1440              97             24                0      RAVDESS        192
    1     4720    7442              30             91                0      CREMA-D          0
    2     9362    2800             119              2                0         TESS          0
    3    11682     480             117              4                0        SAVEE          0


In [19]:
loco = pd.read_parquet("features/phase15_corpus_holdout.parquet")
print(loco[["features","held_out","n_train","n_test","classes_in_test",
            "accuracy","macro_f1_present"]].to_string(index=False))

   features held_out  n_train  n_test  classes_in_test  accuracy  macro_f1_present
handcrafted  RAVDESS    10722    1440                8  0.355556          0.315012
handcrafted  CREMA-D     4720    7442                6  0.329616          0.345696
handcrafted     TESS     9362    2800                7  0.423571          0.425286
handcrafted    SAVEE    11682     480                7  0.412500          0.364529
wav2vec2-L6  RAVDESS    10722    1440                8  0.429167          0.380304
wav2vec2-L6  CREMA-D     4720    7442                6  0.288632          0.314861
wav2vec2-L6     TESS     9362    2800                7  0.391071          0.394855
wav2vec2-L6    SAVEE    11682     480                7  0.352083          0.337630


**0.794 within RAVDESS → 0.380 when RAVDESS is the unseen corpus.** Same 1,440
clips. The only difference is whether training saw other recordings from the
same studio.

So **more than half of the headline is corpus-specific**, not transferable
emotion recognition. Speaker-independent evaluation within one corpus is a much
weaker guarantee than it looks: holding out the voice still shares the
microphone, the room and the prompt set.

**And wav2vec2's advantage does not transfer** — it wins 1 fold of 4 across
corpora, despite winning by 16 points within RAVDESS.

*Metric note:* CREMA-D has no `calm`/`surprised`; TESS and SAVEE have no `calm`.
Macro-F1 over the fixed 8 labels scores absent classes 0 and divides by 8, which
penalises a fold for classes that cannot appear. `macro_f1_present` is the
honest column; both are logged.

---
## 9. Using all four corpora *without* losing accuracy

§7 showed that pooling the external clips into one training set costs 4–11
points. But pooling is not how transfer learning normally works, and the
sequential recipe had never been tested on this project.

Four ways to use the same 12,162 clips, identical frozen test folds. An MLP is
used because two-stage needs a model that can be **continued**, so the control
here is an MLP (0.7938) rather than the logistic-regression headline. The
comparison is internal.

In [20]:
print(pd.read_parquet("features/phase16_two_stage.parquet").to_string(index=False))
print()
print(pd.read_parquet("features/phase16_significance.parquet").to_string(index=False))

             arm  accuracy  macro_f1  seconds  vs_ravdess_only_pp
    ravdess_only  0.799306  0.793830     16.3                0.00
two_stage_frozen  0.785417  0.782180    185.5               -1.17
       two_stage  0.772917  0.768735    185.0               -2.51
          pooled  0.750694  0.747069    195.9               -4.68

             arm  delta_pp  lo_pp  hi_pp     p  significant
          pooled     -4.68  -7.50  -1.92 0.001         True
       two_stage     -2.51  -4.69  -0.41 0.016         True
two_stage_frozen     -1.17  -3.56   1.18 0.343        False


**All four corpora can be used at no statistically measurable cost** — the
frozen-backbone arm sits 1.17 pp below control with a confidence interval
spanning zero (p = 0.34).

The ordering is the finding. Nothing about the data changes across the arms,
only how much influence the 10,722 foreign clips get over the *final decision*:

| Arm | What the foreign data is allowed to do | Cost |
|---|---|---|
| pooled | set the decision boundary directly, outvoting RAVDESS 10:1 | −4.68 pp |
| two-stage, full fine-tune | set the starting point, then get partly overwritten | −2.51 pp |
| **two-stage, frozen backbone** | **decide what features exist; RAVDESS decides the boundaries** | **−1.17 pp** |

**The less the foreign data touches the final decision, the less it costs.**
Monotone across four arms — that is an explanation, not a coincidence.

**It is still not an improvement.** RAVDESS-only remains the highest number in
every comparison in this project. The claim is *free*, not *beneficial*.

### What each arm actually trains on

Same test clips in all four arms — the frozen RAVDESS fold. Only the training
side and the *procedure* differ.

In [21]:
# The merged feature matrix: all 12,162 clips, not the RAVDESS view built in §6.
from ser.run_wav2vec2_transfer import WINNER as W2V_WINNER

tensor_all, index_all = load_cached(EmbeddingConfig(batch_size=8))
feat_all, lab_all = build_matrix(tensor_all, index_all, W2V_WINNER)
idx_all = index_all.set_index("filename")

ext = idx_all.index[idx_all["source"].astype(str) != "RAVDESS"].tolist()
tr0, te0 = get_split("speaker_independent", 0, folds=folds)

print(f"merged matrix {feat_all.shape}")
print(f"fold 0 - RAVDESS train {len(tr0):,} | RAVDESS test {len(te0):,} | external {len(ext):,}\n")
print(pd.DataFrame([
    {"arm": "ravdess_only",     "stage 1": "-",
     "stage 2": f"fit on {len(tr0):,} RAVDESS", "test": f"{len(te0)} RAVDESS"},
    {"arm": "pooled",           "stage 1": "-",
     "stage 2": f"fit on {len(tr0)+len(ext):,} (all 4 corpora at once)", "test": f"{len(te0)} RAVDESS"},
    {"arm": "two_stage",        "stage 1": f"fit on {len(tr0)+len(ext):,} (all 4)",
     "stage 2": f"CONTINUE on {len(tr0):,} RAVDESS", "test": f"{len(te0)} RAVDESS"},
    {"arm": "two_stage_frozen", "stage 1": f"fit on {len(tr0)+len(ext):,} (all 4)",
     "stage 2": f"FREEZE hidden layer, relearn output on {len(tr0):,} RAVDESS", "test": f"{len(te0)} RAVDESS"},
]).to_string(index=False))

merged matrix (12162, 768)
fold 0 - RAVDESS train 1,140 | RAVDESS test 300 | external 10,722

             arm               stage 1                                              stage 2        test
    ravdess_only                     -                                 fit on 1,140 RAVDESS 300 RAVDESS
          pooled                     -                fit on 11,862 (all 4 corpora at once) 300 RAVDESS
       two_stage fit on 11,862 (all 4)                            CONTINUE on 1,140 RAVDESS 300 RAVDESS
two_stage_frozen fit on 11,862 (all 4) FREEZE hidden layer, relearn output on 1,140 RAVDESS 300 RAVDESS


### The code, run live on fold 0

`warm_start=True` is what makes stage 2 possible: calling `fit` again resumes
from the existing weights instead of reinitialising them. That is the whole
mechanism — pretrain on everything, then keep training on RAVDESS alone.

In [22]:
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

X_tr_all, y_tr_all = feat_all.loc[tr0 + ext], lab_all.loc[tr0 + ext]   # all 4 corpora
X_tr,     y_tr     = feat_all.loc[tr0],       lab_all.loc[tr0]         # RAVDESS only
X_te,     y_te     = feat_all.loc[te0],       lab_all.loc[te0]         # RAVDESS test

def f1(pred): return f1_score(y_te, pred, average="macro", zero_division=0)

# --- arm 1: RAVDESS only -------------------------------------------------
m1 = MLPClassifier((256,), max_iter=180, random_state=SEED)
m1.fit(X_tr, y_tr)

# --- arm 2: pooled, everything in one go ---------------------------------
m2 = MLPClassifier((256,), max_iter=180, random_state=SEED)
m2.fit(X_tr_all, y_tr_all)

# --- arm 3: two-stage, full fine-tune ------------------------------------
m3 = MLPClassifier((256,), max_iter=60, random_state=SEED, warm_start=True)
m3.fit(X_tr_all, y_tr_all)          # stage 1: learn from all 12,162
m3.set_params(max_iter=120)
m3.fit(X_tr, y_tr)                  # stage 2: CONTINUE on RAVDESS only

# --- arm 4: two-stage, frozen backbone -----------------------------------
m4 = MLPClassifier((256,), max_iter=60, random_state=SEED)
m4.fit(X_tr_all, y_tr_all)          # stage 1: same pretraining
relu = lambda A: np.maximum(A @ m4.coefs_[0] + m4.intercepts_[0], 0)
head = LogisticRegression(C=0.01, max_iter=2000, random_state=SEED)
head.fit(relu(X_tr.to_numpy()), y_tr)     # stage 2: hidden layer FROZEN,
                                          # only the output layer relearned

print("fold 0 only (the full 5-fold numbers are in the table above)\n")
for name, pred in [("ravdess_only",     m1.predict(X_te)),
                   ("pooled",           m2.predict(X_te)),
                   ("two_stage",        m3.predict(X_te)),
                   ("two_stage_frozen", head.predict(relu(X_te.to_numpy())))]:
    print(f"  {name:<18s} macro-F1 = {f1(pred):.4f}")

fold 0 only (the full 5-fold numbers are in the table above)

  ravdess_only       macro-F1 = 0.8157
  pooled             macro-F1 = 0.7178
  two_stage          macro-F1 = 0.7691
  two_stage_frozen   macro-F1 = 0.7517


Single-fold numbers wobble — 300 test clips from 5 speakers. The table above is
the pooled 5-fold result and is the one to quote. This cell exists to show the
mechanism, not to produce the headline.

The difference between arms 3 and 4 is one line: arm 3 lets stage 2 move
**every** weight, arm 4 freezes the hidden layer and moves **only the output
layer**. That single change is worth 1.3 pp, and it is why the frozen version
is the one that ends up indistinguishable from not using the extra data at all.

---
## 10. Choosing the configuration honestly

A caveat about our own method, and the fix.

Earlier work chose the winning configuration — layer, pooling, normalisation,
head — by comparing **pooled test-fold scores** across ~18 candidates and taking
the best. That is selection on the test set: the maximum of a noisy set is
optimistically biased even when each individual estimate is unbiased.

**Nested cross-validation** fixes it. The configuration is chosen by an inner,
speaker-grouped CV *inside each training fold*, then applied once to the outer
held-out fold, which never participates in its own configuration choice.

In [23]:
print(pd.read_parquet("features/phase17_nested_cv.parquet").to_string(index=False))
print("\ntop candidates when scored directly on the test folds:")
print(pd.read_parquet("features/phase17_candidates.parquet").head(6).to_string(index=False))

                  procedure  macro_f1      lo       hi  optimism_pp
selected on test (Phase 13)  0.828102     NaN      NaN         0.61
       nested CV (unbiased)  0.822041 0.78114 0.858495         0.00

top candidates when scored directly on the test folds:
 layer  pooling normalisation  macro_f1
     6 mean_std   per_speaker  0.828102
     6     mean   per_speaker  0.802825
     7 mean_std   per_speaker  0.795995
     3 mean_std   per_speaker  0.791414
     5 mean_std   per_speaker  0.790955
     9 mean_std   per_speaker  0.786260


**Two results.**

**The selection bias was small — 0.61 pp.** The earlier method was optimistic,
but only mildly; it was sound even though the reporting was loose.

**The honest number is 2.77 pp *higher* than the old headline.** The candidate
set contained a combination never previously tried: `mean_std` pooling at
`C=0.01`. Earlier sweeps tested pooling at the default `C=1.0` (and picked
`mean`), then tested `C=0.01` only with `mean`. The cell in between is worth
2.8 pp.

That is a real lesson about one-axis-at-a-time sweeping: it finds the best value
of each axis *given the others*, which is not the same as the best combination.

**The choice is stable** — four of five folds independently picked
`L6 · mean_std · per-speaker` without seeing their own test data. A selection
driven by noise would scatter.

### Final headline

**0.8243 accuracy / 0.8220 macro-F1**, 95% CI [0.781, 0.859] — nested CV,
pooled out-of-fold, speaker-independent within RAVDESS.

---
## 11. What the model gets wrong

In [24]:
print("per class:")
print(pd.read_parquet("features/phase14c_per_class.parquet").round(3).to_string())
print("\nby gender:")
print(pd.read_parquet("features/phase14c_gender.parquet").round(4).to_string(index=False))
print("\nby intensity:")
print(pd.read_parquet("features/phase14c_intensity.parquet").round(4).to_string(index=False))

per class:
     emotion  precision  recall     f1  support
0    neutral      0.742   0.719  0.730       96
1       calm      0.769   0.849  0.807      192
2      happy      0.746   0.750  0.748      192
3        sad      0.682   0.635  0.658      192
4      angry      0.867   0.880  0.873      192
5    fearful      0.820   0.828  0.824      192
6    disgust      0.872   0.854  0.863      192
7  surprised      0.866   0.839  0.852      192

by gender:
gender   n  accuracy  macro_f1  f1_lo  f1_hi
female 720    0.8431    0.8386 0.7953 0.8822
  male 720    0.7556    0.7501 0.6728 0.8101

by intensity:
intensity_label   n  accuracy  macro_f1  f1_lo  f1_hi           scope
         normal 768    0.7617    0.7617 0.7116 0.8096    all 8 labels
         strong 672    0.8423    0.7385 0.6953 0.8740    all 8 labels
         normal 672    0.7679    0.6814 0.6377 0.7245 7 shared labels
         strong 672    0.8423    0.7385 0.6953 0.8740 7 shared labels


In [25]:
cm = pd.read_parquet("features/phase14c_confusion.parquet")
print("confusion matrix (rows = true, cols = predicted):")
print(cm.to_string())

confusion matrix (rows = true, cols = predicted):
predicted  neutral  calm  happy  sad  angry  fearful  disgust  surprised
true                                                                    
neutral         69    11      7    8      1        0        0          0
calm             9   163      2   17      0        1        0          0
happy            4     3    144    7     10        8        4         12
sad              7    31      5  122      2       16        8          1
angry            1     0      7    2    169        1        5          7
fearful          2     2      8   10      5      159        3          3
disgust          1     2      5   12      4        2      164          2
surprised        0     0     15    1      4        7        4        161


- **`sad` is the weak class** (0.658), not `neutral` as in the handcrafted
  phases. The mechanism changed: it is confusability, not class imbalance.
  `sad ↔ calm` alone causes 48 of 289 errors — both quiet and flat.
- **Gender gap 8.8 pp, worse on male.** The handcrafted model had 12.8 pp in the
  same direction. Better features shrank it; they did not remove it.
- **Better on acted delivery than normal** — 0.739 vs 0.681 on equal class
  footing. Nobody on a real support call is performing, so 0.681 is the more
  honest deployment figure.

---
## 12. How confident are these numbers?

With **24 test speakers**, a small difference is easily luck. `ser.stats`
resamples **speakers, not clips** — 60 clips from one actor succeed or fail
together, and treating them as independent narrows intervals in exactly the
direction that manufactures findings.

In [26]:
from ser.stats import cluster_bootstrap_ci

frames = []
for f in range(5):
    tr, te = get_split("speaker_independent", f, folds=folds)
    clf = LogisticRegression(max_iter=2000, random_state=SEED)
    clf.fit(X.loc[tr].to_numpy(), y.loc[tr].to_numpy())
    frames.append(pd.DataFrame({
        "y_true": y.loc[te].to_numpy(),
        "y_pred": clf.predict(X.loc[te].to_numpy()),
        "speaker": idx.set_index("filename").loc[te, "speaker_id"].to_numpy()}))
allp = pd.concat(frames, ignore_index=True)

ci = cluster_bootstrap_ci(allp.y_true, allp.y_pred, allp.speaker, "macro_f1")
print(f"macro-F1 {ci['point']:.4f}   95% CI [{ci['lo']:.4f}, {ci['hi']:.4f}]"
      f"   (n speakers = {ci['n_speakers']})")

macro-F1 0.7944   95% CI [0.7479, 0.8355]   (n speakers = 24)


**±4.5 points.** That is the width 24 speakers buy, and it changed two things
already written up as results: Phase 5's +0.22 pp tuning gain and Phase 4's
−1.25 pp framing loss are both inside the noise. It also stopped us reporting
logistic regression as beating SVM-RBF, when the 0.88 pp gap has p = 0.44.

**Rule of thumb for this dataset: a difference under ~2 points is not a result.**

---
## 13. Summary and how to extend this

In [27]:
summary = pd.DataFrame([
    ["MFCC-40 baseline",                      "RAVDESS, unseen speakers", 0.515, 0.498],
    ["Handcrafted, swept + tuned",            "RAVDESS, unseen speakers", 0.639, 0.631],
    ["+ all 3 external corpora in training",  "RAVDESS, unseen speakers", 0.524, 0.520],
    ["wav2vec2 L6 + per-speaker + linear",    "RAVDESS, unseen speakers", 0.799, 0.794],
    ["  same, config chosen by NESTED CV",    "RAVDESS, unseen speakers", 0.824, 0.822],
    ["all 4 corpora, frozen-backbone transfer","RAVDESS, unseen speakers", 0.785, 0.782],
    ["wav2vec2, normal-intensity clips only", "RAVDESS, unseen speakers", 0.768, 0.681],
    ["wav2vec2, LEAVE-ONE-CORPUS-OUT",        "RAVDESS entirely unseen",  0.429, 0.380],
], columns=["approach","test condition","accuracy","macro_f1"])
summary

,approach,test condition,accuracy,macro_f1
0,MFCC-40 baseline,"RAVDESS, unseen speakers",0.515,0.498
1,"Handcrafted, swept + tuned","RAVDESS, unseen speakers",0.639,0.631
2,+ all 3 external corpora in training,"RAVDESS, unseen speakers",0.524,0.520
3,wav2vec2 L6 + per-speaker + linear,"RAVDESS, unseen speakers",0.799,0.794
4,"same, config chosen by NESTED CV","RAVDESS, unseen speakers",0.824,0.822
5,"all 4 corpora, frozen-backbone transfer","RAVDESS, unseen speakers",0.785,0.782
6,"wav2vec2, normal-intensity clips only","RAVDESS, unseen speakers",0.768,0.681
7,"wav2vec2, LEAVE-ONE-CORPUS-OUT",RAVDESS entirely unseen,0.429,0.380


### Adding a new experiment

Swap the model; keep everything else identical. That is the entire contract, and
it is what makes a new result comparable to every result already in the log.

```python
from ser.splits import get_split
from ser.evaluate import evaluate_pooled

per_fold = []
for fold in range(5):
    train_files, test_files = get_split("speaker_independent", fold)
    model = YourModel()                      # <- the only line that is yours
    model.fit(X.loc[train_files], y.loc[train_files])
    pred = model.predict(X.loc[test_files])
    per_fold.append((y.loc[test_files], pred, meta_for(test_files)))

evaluate_pooled(per_fold, "speaker_independent",
                model="your-model", feature_config="your-features",
                notes="what you were testing")
```

### Building an attention model on top of this

**The cached embeddings cannot be used for this.** `extract()` pools each clip to
a mean and standard deviation over time — the time axis is gone, and pooling is
exactly what attention is meant to replace. Use `extract_frames()` instead,
which yields per-frame states:

```python
from ser.embeddings import extract_frames, EmbeddingConfig

for states, lengths, names in extract_frames(meta, EmbeddingConfig(), layer=6):
    # states: (batch, max_frames, 768), zero-padded on the right
    # lengths: real frame count per clip — USE IT
    mask = torch.arange(states.shape[1])[None, :] < torch.tensor(lengths)[:, None]
```

**Mask the padding.** Clips differ in length; an attention layer that ignores
`lengths` will happily attend to silence that is not in the audio.

**Expect overfitting.** There are only ~1,140 training clips per fold. A
*linear* classifier beat XGBoost by 13 points on these features, which is a
strong hint that the dataset does not support a high-capacity model. Any
attention architecture should be small, regularised, and validated with the
inner-CV pattern used in §10 rather than tuned against the test folds.

### Work not yet attempted

| Idea | Why |
|---|---|
| **Augmentation** | Generate extra training examples by slightly altering existing clips — small speed changes, pitch changes, or added background noise. Applied to training folds only, never to test. Untested in this project. |
| **Speaker-adversarial training** | The representation identifies which of the 24 actors is speaking with 94.9% accuracy (§11). A second output head can be trained to predict the speaker, with its gradient reversed, so the main model is pushed to discard speaker identity while keeping emotion. Directly attacks the largest known weakness. |
| **WavLM / HuBERT** | Alternative pretrained encoders of the same size. One extraction pass each (~1 hour of CPU). Published gains on emotion tasks are 1-3 points, which is below what 24 test speakers can reliably detect. |

### Where everything lives

| | |
|---|---|
| `docs/` | one findings document per phase, with the reasoning |
| `results/results.csv` | every run ever, 900+ rows |
| `splits/folds.csv` | the frozen contract — **never edit** |
| `figures/` | plots for the deck |
| `ser/` | all logic; notebooks import from here |